<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

### Cosmos3 Reasoner inference with NVIDIA NIM

This notebook:
1. Launches the [Cosmos 3 Reasoner NIM](https://catalog.ngc.nvidia.com/orgs/nim/teams/nvidia/containers/cosmos3-reasoner) container, a prebuilt, optimized, OpenAI-compatible server.
2. Waits for it to become ready.
3. Sends the same image and video reasoning requests with the `openai` client.

Compared to the vLLM notebook, there is no Python environment or CUDA-pairing to set up — the NIM container ships everything. The container serves two sizes, selected with `NIM_MODEL_SIZE`:

| `NIM_MODEL_SIZE` | Served model name |
| --- | --- |
| `nano` (default) | `nvidia/cosmos3-nano-reasoner` |
| `super` | `nvidia/cosmos3-super-reasoner` |

You can also try this same NIM interactively in your browser on the [cosmos3-nano-reasoner build page](https://build.nvidia.com/nvidia/cosmos3-nano-reasoner). See the [Cosmos Reason 3 NIM API reference](https://docs.nvidia.com/nim/vision-language-models/1.7.0/examples/cosmos-reason3/api.html) for the full request reference.

## 1. Launch the NIM container

You need an `NGC_API_KEY` (from [build.nvidia.com](https://build.nvidia.com/nvidia/cosmos3-nano-reasoner) or [NGC](https://catalog.ngc.nvidia.com/orgs/nim/teams/nvidia/containers/cosmos3-reasoner)) set in the environment **before** starting Jupyter, or set it from Python with `os.environ["NGC_API_KEY"] = "nvapi-..."`. Docker must also be logged in to `nvcr.io` (`docker login nvcr.io`, username `$oauthtoken`, password = your key).

The first launch downloads and caches the model into `~/.cache/nim`, which can take a while; later launches start quickly.

In [ ]:
from pathlib import Path
import os
import subprocess


def find_repo_root() -> Path:
    try:
        return Path(
            subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        ).resolve()
    except Exception:
        return Path.cwd().resolve()


COSMOS_ROOT = find_repo_root()
COSMOS_REASONER_ASSETS = COSMOS_ROOT / "cookbooks" / "cosmos3" / "reasoner" / "assets"

assert COSMOS_REASONER_ASSETS.exists(), COSMOS_REASONER_ASSETS


def asset_path(name: str) -> Path:
    path = COSMOS_REASONER_ASSETS / name
    if not path.exists():
        raise FileNotFoundError(path)
    return path


import base64
import mimetypes


def asset_data_uri(name: str) -> str:
    """Read a local asset and return a base64 ``data:`` URI.

    The NIM container does not see the host filesystem, so local images and
    videos are inlined as base64 data URIs rather than passed as ``file://``
    paths. Public URLs work too — see the Image Caption example below.
    """
    path = asset_path(name)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


print("cosmos root:", COSMOS_ROOT)
print("Reasoner assets:", COSMOS_REASONER_ASSETS)


In [ ]:
%%bash
set -euo pipefail

: "${NGC_API_KEY:?Set NGC_API_KEY (from build.nvidia.com / NGC) before launching}"

export CONTAINER_NAME="${CONTAINER_NAME:-nvidia-cosmos3-reasoner}"
export IMG_NAME="${IMG_NAME:-nvcr.io/nim/nvidia/cosmos3-reasoner:1.7.0}"
# Set NIM_MODEL_SIZE=super for the larger Cosmos3-Super-Reasoner.
export NIM_MODEL_SIZE="${NIM_MODEL_SIZE:-nano}"
export LOCAL_NIM_CACHE="${LOCAL_NIM_CACHE:-$HOME/.cache/nim}"
export NIM_PORT="${NIM_PORT:-8000}"
mkdir -p "$LOCAL_NIM_CACHE"

# The container name must be free. If a previous run is still up, stop it first:
#   docker stop nvidia-cosmos3-reasoner
# Detached (-d) so the notebook keeps control; logs are read by the next cell.
docker run -d --rm --name="$CONTAINER_NAME" \
  --runtime=nvidia \
  --gpus all \
  --shm-size=32GB \
  -e NGC_API_KEY="$NGC_API_KEY" \
  -e NIM_MODEL_SIZE="$NIM_MODEL_SIZE" \
  -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
  -u "$(id -u)" \
  -p "${NIM_PORT}:8000" \
  "$IMG_NAME"

echo "NIM container '$CONTAINER_NAME' (size=$NIM_MODEL_SIZE) launching on port $NIM_PORT"


## 2. Wait for the server

The next cell polls the NIM readiness endpoint and streams recent container logs. The first run downloads the model, so allow several minutes.

In [ ]:
%%bash
set -euo pipefail

PORT="${NIM_PORT:-8000}"
CONTAINER_NAME="${CONTAINER_NAME:-nvidia-cosmos3-reasoner}"

echo "Waiting for NIM server on port ${PORT} (first run downloads the model)..."

for i in $(seq 1 3600); do
  if curl -fsS "http://127.0.0.1:${PORT}/v1/health/ready" >/dev/null 2>&1; then
    echo "NIM server is ready."
    exit 0
  fi
  if ! docker ps --format '{{.Names}}' | grep -q "^${CONTAINER_NAME}$"; then
    echo "Container '${CONTAINER_NAME}' is no longer running. Recent logs:"
    docker logs --tail 120 "$CONTAINER_NAME" 2>&1 || true
    exit 1
  fi
  sleep 2
done

echo "Timed out waiting for NIM server. Recent logs:"
docker logs --tail 120 "$CONTAINER_NAME" 2>&1 || true
exit 1


## 3. Query the model

Local images and videos are sent as base64 data URIs (the container does not see the host filesystem). Each request resolves the served model dynamically with `client.models.list()`, so the examples work unchanged for both the `nano` and `super` sizes.

### Image Caption

In [ ]:
import openai
from IPython.display import Image, display

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id

image_url = (
    "https://github.com/nvidia-cosmos/cosmos-dependencies/raw/refs/heads/assets/cosmos3/inputs/vision/robot_153.jpg"
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": "Caption the image in detail."},
            ],
        }
    ],
    max_tokens=4096,
    seed=0,
)
display(Image(url=image_url, width=512))
print(response.choices[0].message.content)

### Video Caption

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display

prompt = "Describe the video in detail."

# Plain filesystem path (used for display)
video_path = str(asset_path("video_caption.mp4"))
# base64 data URI (used for the model request)
video_url = asset_data_uri(Path(video_path).name)

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

# Display the input video (plain path, not the base64 data URI sent to the model)
display(Video(video_path, embed=True, width=640))

print(response.choices[0].message.content)

### Temporal Localization

In [ ]:
from pathlib import Path
from IPython.display import Video, display

# Plain filesystem path (used for display)
video_path = str(asset_path("temporal_localization_1.mp4"))

display(Video(video_path, embed=True, width=640))

import openai

prompt = (
    """List all action segments in the video. For each detected event, you must determine:

Provide the result in json format with 'seconds' for time depiction for each event. Use keywords 'start', 'end' and 'caption' in the json output. Please list multiple events if applicable.

```json
[
{
  "start": t_start,
  "end": t_end,
  "caption": EVENT1
},
{
  "start": t_start,
  "end": t_end,
  "caption": EVENT2
},
...
]
``` """
)
video_url = asset_data_uri("temporal_localization_1.mp4")

client = openai.OpenAI(
    api_key="not-used",
    base_url="http://localhost:8000/v1",
)

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

print(response.choices[0].message.content)

In [ ]:
from pathlib import Path
from IPython.display import Video, display

# Plain filesystem path (used for display)
video_path = str(asset_path("temporal_localization_2.mp4"))

display(Video(video_path, embed=True, width=640))

#### Event Timeline

In [ ]:
import openai

prompt = (
    "Describe the notable events in the provided video. Provide the result in json format with 'mm:ss.ff' format for time depiction for each event."
    "Use keywords 'start', 'end' and 'caption' in the json output."
)
video_url = asset_data_uri("temporal_localization_2.mp4")

client = openai.OpenAI(
    api_key="not-used",
    base_url="http://localhost:8000/v1",
)

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

print(response.choices[0].message.content)

#### Timestamp Query

In [ ]:
import openai

prompt = """When is "A man in a white sweater walks out of a room carrying a box, closes the door behind him, walks on the floor, and turns left at the end near the wall." depicted in the video? Please provide the result in json format with 'mm:ss.ff' format for time depiction for the event. Use keywords 'start', 'end' in the json output."""

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

print(response.choices[0].message.content)

#### Interval Question

In [ ]:
import openai

prompt = "What happened between 00:05.64 and 00:17.49?"
response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

print(response.choices[0].message.content)

### Embodied Reasoning
#### Robotics Next Action

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display

prompt = "What can be the next immediate action? Answer the question using the following format: <think> Your reasoning. </think> Write your final answer immediately after the </think> tag."

# Plain filesystem path (used for display)
video_path = str(asset_path("robotics_next_action.mp4"))
# base64 data URI (used for the model request)
video_url = asset_data_uri(Path(video_path).name)

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

# Display the input video (plain path, not the base64 data URI sent to the model)
display(Video(video_path, embed=True, width=640))

print(response.choices[0].message.content)

#### Drive Scene Next Action

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display

prompt = "You are an autonomous vehicle planning system. The video depicts the observation from the vehicle's camera. You need to observe the critical objects in the environment and reason your next action and the driving trajectory ahead."

# Plain filesystem path (used for display)
video_path = str(asset_path("drive_scene_next_action.mp4"))
# base64 data URI (used for the model request)
video_url = asset_data_uri(Path(video_path).name)

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

# Display the input video (plain path, not the base64 data URI sent to the model)
display(Video(video_path, embed=True, width=640))

print(response.choices[0].message.content)

#### Robot Planning

In [ ]:
import json
import re
import openai
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id

image_path = str(asset_path("robot_planning.png"))
image_url = asset_data_uri(Path(image_path).name)   # base64 data URI for the model

# Display the input image (scaled down to fit the cell)
preview = PILImage.open(image_path).convert("RGB")
preview.thumbnail((768, 768))
display(preview)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": 'The task is to put flower into the red bottle. Generate a plan consisting of subtasks for accomplish the task.'},
            ],
        }
    ],
    max_tokens=4096,
    seed=0,
)
print(response.choices[0].message.content)

#### Assisted Task Next Action

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display

prompt = """This is the overall task that the agent is trying to complete: "The student exchanges the black ink cartridge of the printer."
            In the video, the agent is trying to follow the instruction (a single step out of many to complete the overall task): "place old ink_cartridge."
            What should be the next action of the agent?
            Answer the question using the following format:
            <think>
            Your reasoning.
            </think>
            Write your final answer immediately after the </think> tag."""

# Plain filesystem path (used for display)
video_path = str(asset_path("assisted_task_next_action.mp4"))
# base64 data URI (used for the model request)
video_url = asset_data_uri(Path(video_path).name)

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

# Display the input video (plain path, not the base64 data URI sent to the model)
display(Video(video_path, embed=True, width=640))

print(response.choices[0].message.content)

### Common Sense Reasoning

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display

prompt = """Can the countertop support the weight of the juicers?
            Answer the question using the following format:

            <think>
            Your reasoning.
            </think>

            Write your final answer immediately after the </think> tag."""

# Plain filesystem path (used for display)
video_path = str(asset_path("common_sense_reasoning.mp4"))
# base64 data URI (used for the model request)
video_url = asset_data_uri(Path(video_path).name)

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")

response = client.chat.completions.create(
    model=client.models.list().data[0].id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        },
    ],
    max_tokens=4096,
    extra_body={"media_io_kwargs": {"video": {"fps": 4.0}}},
)

# Display the input video (plain path, not the base64 data URI sent to the model)
display(Video(video_path, embed=True, width=640))

print(response.choices[0].message.content)

### 2D Grounding

In [ ]:
import json
import re
import openai
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id

image_path = str(asset_path("grounding_2d.png"))
image_url = asset_data_uri(Path(image_path).name)   # base64 data URI for the model

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": "Locate the accurate bounding box of the load as a whole. Return a json."},
            ],
        }
    ],
    max_tokens=4096,
    seed=0,
)
out = response.choices[0].message.content
print(out)


def parse_boxes(text):
    """Pull a JSON array/object of boxes out of the model text (handles ``` fences)."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    m = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    data = json.loads(m.group(0) if m else text)
    return data if isinstance(data, list) else [data]


# Draw boxes; coords are normalized to 0-1000
img = PILImage.open(image_path).convert("RGB")
W, H = img.size
draw = ImageDraw.Draw(img)

for obj in parse_boxes(out):
    box = obj.get("bbox_2d") or obj.get("bbox") or obj.get("box")
    if not box:
        continue
    x1, y1, x2, y2 = box
    x1, x2 = x1 / 1000 * W, x2 / 1000 * W
    y1, y2 = y1 / 1000 * H, y2 / 1000 * H
    draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    label = obj.get("label") or obj.get("name")
    if label:
        draw.text((x1, max(0, y1 - 12)), str(label), fill="red")

# Display scaled down so a large image fits the cell
preview = img.copy()
preview.thumbnail((768, 768))
display(preview)

### Describe Anything

In [ ]:
import json
import re
import openai
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id

image_path = str(asset_path("describe_anything.png"))
image_url = asset_data_uri(Path(image_path).name)   # base64 data URI for the model

# Display the input image (scaled down to fit the cell)
preview = PILImage.open(image_path).convert("RGB")
preview.thumbnail((768, 768))
display(preview)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": 'Please caption the notable attributes in the provided image. List and describe all marked subjects in the image with their categories and detailed captions using a json with keyword "subject_id", "category" and "caption".'},
            ],
        }
    ],
    max_tokens=4096,
    seed=0,
)
print(response.choices[0].message.content)


### Action CoT

#### Trajectory Coordinates

In [ ]:
import json
import re
import openai
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id

image_path = str(asset_path("action_cot_trajectory.png"))
image_url = asset_data_uri(Path(image_path).name)

prompt = """You are given the task "Move the pink bowl to the right". Specify the 2D trajectory your end effector should follow in pixel space. Return the trajectory coordinates in JSON format like this: {"point_2d": [x, y], "label": "gripper trajectory"}.
Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": prompt},
            ],
        }
    ],
    max_tokens=4096,
    temperature=0.6,
    top_p=0.95,
    presence_penalty=0.0,
    extra_body={"top_k": 20, "repetition_penalty": 1.0},
)
out = response.choices[0].message.content
print(out)


def parse_points(text):
    """Grab the JSON list of {point_2d, label} after the </think> tag."""
    if "</think>" in text:
        text = text.split("</think>")[-1]
    text = re.sub(r"```(?:json)?", "", text).strip().strip("`").strip()
    m = re.search(r"\[.*\]", text, re.DOTALL)
    data = json.loads(m.group(0) if m else text)
    return data if isinstance(data, list) else [data]


# Visualize the trajectory (points are in pixel space)
img = PILImage.open(image_path).convert("RGB")
draw = ImageDraw.Draw(img)
W, H = img.size

# coords are normalized to 0-1000 (per-axis) -> scale to pixels
pts = [(o["point_2d"][0] / 1000 * W, o["point_2d"][1] / 1000 * H)
       for o in parse_points(out) if isinstance(o, dict) and "point_2d" in o]
if len(pts) > 1:
    draw.line(pts, fill="lime", width=5)
for i, (x, y) in enumerate(pts):
    r = 12
    draw.ellipse([x - r, y - r, x + r, y + r], fill="red", outline="white", width=3)
    draw.text((x + 14, y - 14), str(i), fill="yellow")
preview = img.copy()
preview.thumbnail((900, 900))
display(preview)

In [ ]:
import json
import re
import openai
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
from IPython.display import display

client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id

image_path = str(asset_path("robot_planning.png"))
image_url = asset_data_uri(Path(image_path).name)

prompt = """You are given the task "Put flower into the red bottle". Specify the 2D trajectory your end effector should follow in pixel space. Return the trajectory coordinates in JSON format like this: {"point_2d": [x, y], "label": "gripper trajectory"}. 
Answer the question using the following format:

<think> Your reasoning. </think>
Write your final answer immediately after the </think> tag.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text", "text": prompt},
            ],
        }
    ],
    max_tokens=4096,
    temperature=0.6,
    top_p=0.95,
    presence_penalty=0.0,
    extra_body={"top_k": 20, "repetition_penalty": 1.0},
)
out = response.choices[0].message.content
print(out)


def parse_points(text):
    """Grab the JSON list of {point_2d, label} after the </think> tag."""
    if "</think>" in text:
        text = text.split("</think>")[-1]
    text = re.sub(r"```(?:json)?", "", text).strip().strip("`").strip()
    m = re.search(r"\[.*\]", text, re.DOTALL)
    data = json.loads(m.group(0) if m else text)
    return data if isinstance(data, list) else [data]


# Visualize the trajectory (points are in pixel space)
img = PILImage.open(image_path).convert("RGB")
draw = ImageDraw.Draw(img)
W, H = img.size

# coords are normalized to 0-1000 (per-axis) -> scale to pixels
pts = [(o["point_2d"][0] / 1000 * W, o["point_2d"][1] / 1000 * H)
       for o in parse_points(out) if isinstance(o, dict) and "point_2d" in o]
if len(pts) > 1:
    draw.line(pts, fill="lime", width=5)
for i, (x, y) in enumerate(pts):
    r = 12
    draw.ellipse([x - r, y - r, x + r, y + r], fill="red", outline="white", width=3)
    draw.text((x + 14, y - 14), str(i), fill="yellow")
preview = img.copy()
preview.thumbnail((900, 900))
display(preview)

#### Driving Scene

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display
client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id
video_path = str(asset_path("action_cot_driving_scene.mp4"))
video_url = asset_data_uri(Path(video_path).name)
prompt = """The video depicts the observation from the vehicle's camera. You need to think step by step and identify the objects in the scene that are critical for safe navigation.
Answer the question using the following format:
<think>
Your reasoning.
</think>
Write your final answer immediately after the </think> tag."""
# Show the input video
display(Video(video_path, embed=True, width=640))
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        }
    ],
    max_tokens=4096,
    temperature=0.6,
    top_p=0.95,
    presence_penalty=0.0,
    extra_body={
        "top_k": 20,
        "repetition_penalty": 1.0,
        "media_io_kwargs": {"video": {"fps": 4.0}},
    },
)
print(response.choices[0].message.content)

### Physical Plausibility Analysis

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display
client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id
video_path = str(asset_path("physical_plausibility.mp4"))
video_url = asset_data_uri(Path(video_path).name)
prompt = """Is this video physically plausible/possible according to your understanding of e.g. object permanence, shape constancy (objects maintain shape over time), continuous trajectories of objects? Assume it is the normal laws of physics.
Your answer should be based on the events in the video and ignore the quality of the simulation engine. The rising wall is part of the experiment setup and should not be judged for plausibility.
(A) Possible
(B) Impossible"""
# Show the input video
display(Video(video_path, embed=True, width=640))
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        }
    ],
    max_tokens=4096,
    extra_body={
        "media_io_kwargs": {"video": {"fps": 4.0}},
    },
)
print(response.choices[0].message.content)

### Situation Understanding

In [ ]:
import openai
from pathlib import Path
from IPython.display import Video, display
client = openai.OpenAI(api_key="not-used", base_url="http://localhost:8000/v1")
MODEL = client.models.list().data[0].id
video_path = str(asset_path("situation_understanding.mp4"))
video_url = asset_data_uri(Path(video_path).name)
prompt = "What is the person doing with the skillet? What will the person likely do next in this situation?"
# Show the input video
display(Video(video_path, embed=True, width=640))
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_url}},
                {"type": "text", "text": prompt},
            ],
        }
    ],
    max_tokens=4096,
    extra_body={
        "media_io_kwargs": {"video": {"fps": 4.0}},
    },
)
print(response.choices[0].message.content)